In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import random

In [ ]:
# Original (cluster, old paper model) inputs, kept for reference:

PROJECT_ROOT = Path("/path/to/ainu_root")
IMAGE_PATH = PROJECT_ROOT / "data" / "human_data"
output_dir = PROJECT_ROOT / "features_analysis" / "tsne" / "mrd_newmodel_wo_AML12_14"
output_dir.mkdir(parents=True, exist_ok=True)
images_features = IMAGE_PATH / "cell_features.csv"

# New model (results/4-CT-AML-PB-BM, trained without AML12/AML14).
# One file with the held-out AML/Control cells + all MRD and MDS cells, embedded and t-SNE'd jointly
# (made by embed_newmodel_with_MRD_MDS.py)
embeddings_file = PROJECT_ROOT / "embeddings_with_predictions_newmodel_MRD_MDS.csv"

features_df = pd.read_csv(images_features, header=[0], low_memory=False)
fused_df = pd.read_csv(embeddings_file, header=[0], low_memory=False)
print(fused_df.shape)


In [ ]:
# Labels used by the plots below:
#   AML / Control = the new model's held-out test cells
#   MRD           = every cell of the MRD cohort (MRD column filled: negative, low-positive,
#                   positive, unspecific and AML14)
#   preleukemia   = MDS, CMML, CCUS and AML12 (CEL), i.e. cells without an MRD annotation
is_ref = fused_df["cohort"] == "Reference test (AML/Control)"
fused_df["new_labels"] = np.where(
    is_ref, fused_df["new_labels"],
    np.where(fused_df["MRD"].notna(), "MRD", "preleukemia")
)
fused_df["prob_positive"] = fused_df["prob_AML"]   # the plots expect the AML probability under this name
print(fused_df["new_labels"].value_counts())


In [ ]:
# The two embedding files are no longer fused: the new embeddings are one jointly computed file.
print(f"Fused shape: {fused_df.shape}")
print("\nt-SNE range per label:")
print(fused_df.groupby("new_labels")[["tsne_1", "tsne_2"]].agg(["min", "max"]).round(1))

fused_path = output_dir / "embeddings_newmodel_fused.csv"
fused_df.to_csv(fused_path, index=False)
print(f"\nSaved: {fused_path}")

# Data-driven y-limits (the old hard-coded -20..20 would hide most of the new, jointly computed map)
YLIM = (float(np.floor(fused_df["tsne_2"].min()) - 2), float(np.ceil(fused_df["tsne_2"].max()) + 2))
print("YLIM:", YLIM)


In [ ]:
# Merge: probability_df File == features_df File.1
df_merged = fused_df.merge(
    features_df,
    left_on="File",
    right_on="File.1",
    how="inner",
    suffixes=("_A", "_B")
)

print(df_merged.shape)
df_merged.head()

# Tsne plotting 

In [ ]:
print(df_merged["new_labels_A"].value_counts(dropna=False))


In [ ]:
#"preleukemia": "#F4A582",
#"MRD": "#92C5DE",


In [ ]:
import matplotlib.pyplot as plt

label_col = "new_labels_A"

print("Using label column:", label_col)

required_cols = ["tsne_1", "tsne_2", label_col]
missing_cols = [c for c in required_cols if c not in df_merged.columns]

if missing_cols:
    raise ValueError(f"Missing columns: {missing_cols}")

print("\nLabel counts before removing NaNs:")
print(df_merged[label_col].value_counts(dropna=False))

before_rows = len(df_merged)

plot_df = df_merged.dropna(subset=["tsne_1", "tsne_2", label_col]).copy()

after_rows = len(plot_df)

print("\nRows before filtering:", before_rows)
print("Rows after filtering:", after_rows)
print("Rows removed because of NaNs:", before_rows - after_rows)

print("\nLabel counts after removing NaNs:")
print(plot_df[label_col].value_counts(dropna=False))

print("\nClasses plotted:")
print(plot_df[label_col].unique())

color_map = {
    "AML": "#B2182B",
    "Control": "#2166AC",
    "MRD": "#92C5DE"
}

default_colors = plt.cm.tab10.colors
classes = sorted(plot_df[label_col].unique())

plt.figure(figsize=(16.2, 14.2))

for i, cls in enumerate(classes):
    class_df = plot_df[plot_df[label_col] == cls]

    color = color_map.get(cls, default_colors[i % len(default_colors)])

    # Faded individual cells
    plt.scatter(
        class_df["tsne_1"],
        class_df["tsne_2"],
        color=color,
        alpha=0.3,
        s=18,
        #label=f"{cls}",
        zorder=1
    )

    # Class mean
    mean_x = class_df["tsne_1"].mean()
    mean_y = class_df["tsne_2"].mean()

    plt.scatter(
        mean_x,
        mean_y,
        color=color,
        alpha=0.8,
        s=260,
        edgecolor="black",
        linewidth=1.4,
        marker="o",
        label=f"{cls} mean",
        zorder=10
    )

    print(f"\n{cls}")
    print(f"n cells: {len(class_df)}")
    print(f"mean tsne_1: {mean_x:.3f}")
    print(f"mean tsne_2: {mean_y:.3f}")

plt.xlabel("tSNE 1")
plt.ylabel("tSNE 2")
plt.title(f"tSNE colored by {label_col}")

#plt.xlim(-25, 25)
plt.ylim(*YLIM)

plt.legend(
   bbox_to_anchor=(0.5, -0.2),
    loc="lower center",
    frameon=False,
    ncol=3
)
plt.savefig(
    output_dir / "tsne.pdf",
    bbox_inches="tight",
    dpi=300
)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

label_col = "new_labels_A"
patient_col = "patient"

plot_df = df_merged.dropna(subset=["tsne_1", "tsne_2", label_col, patient_col]).copy()

plot_df[label_col] = plot_df[label_col].astype(str).str.strip()
plot_df[patient_col] = plot_df[patient_col].astype(str).str.strip()

classes_to_plot = ["AML", "Control", "MRD"]
default_colors = plt.cm.tab20.colors

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5), sharey=True)

for ax, class_name in zip(axes, classes_to_plot):
    class_plot_df = plot_df[plot_df[label_col] == class_name].copy()

    # Restart patient colors for each plot
    class_patients = sorted(class_plot_df[patient_col].unique())
    patient_color_map = {
        patient: default_colors[i % len(default_colors)]
        for i, patient in enumerate(class_patients)
    }

    for patient in class_patients:
        patient_df = class_plot_df[class_plot_df[patient_col] == patient]

        color = patient_color_map[patient]

        # Faded individual points - no legend
        ax.scatter(
            patient_df["tsne_1"],
            patient_df["tsne_2"],
            color=color,
            alpha=0.3,
            s=18,
            zorder=1
        )

        mean_x = patient_df["tsne_1"].mean()
        mean_y = patient_df["tsne_2"].mean()

        # Mean point on top - this is what goes in the legend
        ax.scatter(
            mean_x,
            mean_y,
            color=color,
            alpha=1.0,
            s=260,
            edgecolor="black",
            linewidth=1.4,
            marker="o",
            zorder=10,
            label=patient
        )

    ax.set_title(class_name)
    ax.set_xlabel("tSNE 1")
    
    ax.set_ylim(*YLIM)
    #ax.set_xlim(-30, 30)
    
    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))

    ax.legend(
        by_label.values(),
        by_label.keys(),
        loc="upper center",
        bbox_to_anchor=(0.5, -0.18),
        ncol=min(len(by_label), 4),
        frameon=False,
        markerscale=0.45
    )

axes[0].set_ylabel("tSNE 2")

plt.savefig(
    output_dir / "tsne_patients.pdf",
    bbox_inches="tight",
    dpi=300
)
#plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

label_col = "new_labels_A"
patient_col = "patient"
prob_col = "prob_positive"

plot_df = df_merged.dropna(
    subset=["tsne_1", "tsne_2", label_col, patient_col, prob_col]
).copy()

plot_df[label_col] = plot_df[label_col].astype(str).str.strip()
plot_df[patient_col] = plot_df[patient_col].astype(str).str.strip()
plot_df[prob_col] = plot_df[prob_col].astype(float)

classes_to_plot = ["AML", "Control", "preleukemia"]

# Blue = low probability, red = high probability
cmap = plt.cm.RdBu_r
norm = Normalize(vmin=0, vmax=1)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(18, 4.5),
    sharey=True
)

for ax, class_name in zip(axes, classes_to_plot):
    class_plot_df = plot_df[
        plot_df[label_col] == class_name
    ].copy()

    class_patients = sorted(
        class_plot_df[patient_col].unique()
    )

    # Plot all individual cells, colored by prob_positive
    ax.scatter(
        class_plot_df["tsne_1"],
        class_plot_df["tsne_2"],
        c=class_plot_df[prob_col],
        cmap=cmap,
        norm=norm,
        alpha=0.35,
        s=18,
        edgecolors="none",
        zorder=1
    )

    # Plot and label each patient mean
    for patient in class_patients:
        patient_df = class_plot_df[
            class_plot_df[patient_col] == patient
        ]

        mean_x = patient_df["tsne_1"].mean()
        mean_y = patient_df["tsne_2"].mean()
        mean_prob = patient_df[prob_col].mean()

        # Patient mean position colored by mean prob_positive
        ax.scatter(
            mean_x,
            mean_y,
            c=[mean_prob],
            cmap=cmap,
            norm=norm,
            alpha=1.0,
            s=260,
            edgecolor="black",
            linewidth=1.4,
            marker="o",
            zorder=10
        )

        # Patient name next to the mean point
#        ax.annotate(
#            patient,
#            xy=(mean_x, mean_y),
#            xytext=(6, 4),
#            textcoords="offset points",
#            fontsize=8,
#            fontweight="bold",
#            color="black",
#            zorder=11
#        )

    ax.set_title(class_name)
    ax.set_xlabel("tSNE 1")
    ax.set_ylim(*YLIM)

axes[0].set_ylabel("tSNE 2")

# Shared probability colorbar
sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])

cbar = fig.colorbar(
    sm,
    ax=axes,
    fraction=0.025,
    pad=0.02
)
cbar.set_label("AML probability")

plt.savefig(
    output_dir / "tsne_patients_prob_positive_labeled2.pdf",
    bbox_inches="tight",
    dpi=300
)

plt.show()

# one tnse colored by probability 

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

label_col = "new_labels_A"
patient_col = "patient"
prob_col = "prob_positive"

# Change this if the preleukemic class has a different label
preleukemia_label = "MRD"

required_cols = [
    "tsne_1",
    "tsne_2",
    label_col,
    patient_col,
    prob_col
]

missing_cols = [
    col for col in required_cols
    if col not in df_merged.columns
]

if missing_cols:
    raise ValueError(f"Missing columns: {missing_cols}")

plot_df = df_merged.dropna(
    subset=required_cols
).copy()

plot_df[label_col] = (
    plot_df[label_col]
    .astype(str)
    .str.strip()
)

plot_df[patient_col] = (
    plot_df[patient_col]
    .astype(str)
    .str.strip()
)

plot_df[prob_col] = (
    plot_df[prob_col]
    .astype(float)
)

# Blue = low AML probability
# Red = high AML probability
cmap = plt.cm.RdBu_r
norm = Normalize(vmin=0, vmax=1)

fig, ax = plt.subplots(figsize=(5, 4))

# --------------------------------------------------
# Individual cells colored by prob_positive
# --------------------------------------------------
ax.scatter(
    plot_df["tsne_1"],
    plot_df["tsne_2"],
    c=plot_df[prob_col],
    cmap=cmap,
    norm=norm,
    alpha=0.30,
    s=18,
    edgecolors="none",
    zorder=1
)

# --------------------------------------------------
# Overall mean for AML and Control
# --------------------------------------------------
for class_name in ["AML", "Control"]:
    class_df = plot_df[
        plot_df[label_col] == class_name
    ]

    if class_df.empty:
        continue

    mean_x = class_df["tsne_1"].mean()
    mean_y = class_df["tsne_2"].mean()
    mean_prob = class_df[prob_col].mean()

    ax.scatter(
        mean_x,
        mean_y,
        c=[mean_prob],
        cmap=cmap,
        norm=norm,
        s=300,
        edgecolor="black",
        linewidth=1.6,
        marker="o",
        zorder=10,
        label=f"{class_name} overall mean"
    )

# --------------------------------------------------
# Individual patient means for the preleukemic group
# --------------------------------------------------
preleukemia_df = plot_df[
    plot_df[label_col] == preleukemia_label
].copy()

patient_means = (
    preleukemia_df
    .groupby(patient_col, as_index=False)
    .agg(
        mean_tsne_1=("tsne_1", "mean"),
        mean_tsne_2=("tsne_2", "mean"),
        mean_prob_positive=(prob_col, "mean")
    )
    .sort_values(patient_col)
)

for _, row in patient_means.iterrows():
    ax.scatter(
        row["mean_tsne_1"],
        row["mean_tsne_2"],
        c=[row["mean_prob_positive"]],
        cmap=cmap,
        norm=norm,
        s=260,
        edgecolor="black",
        linewidth=1.4,
        marker="s",
        zorder=10,
        label=row[patient_col]
    )

ax.set_xlabel("tSNE 1")
ax.set_ylabel("tSNE 2")
ax.set_title("tSNE colored by AML probability")

# ax.set_xlim(-25, 25)
ax.set_ylim(*YLIM)

# --------------------------------------------------
# Legend for AML/Control overall means and
# individual preleukemic patient means
# --------------------------------------------------
#ax.legend(
#    title="Mean positions",
#    bbox_to_anchor=(1.28, 1),
#    loc="upper left",
#    frameon=False,
#    fontsize=8,
#    title_fontsize=9,
#    markerscale=0.55
#)

# --------------------------------------------------
# Probability colorbar
# --------------------------------------------------
sm = ScalarMappable(
    norm=norm,
    cmap=cmap
)
sm.set_array([])

cbar = fig.colorbar(
    sm,
    ax=ax,
    pad=0.03
)
cbar.set_label("AML probability")

plt.savefig(
    output_dir / "tsne_probability_preleukemia_patient_means.pdf",
    bbox_inches="tight",
    dpi=300
)

plt.show()

# one tnse colored by patient

In [ ]:
import matplotlib.pyplot as plt

label_col = "new_labels_A"
patient_col = "patient"

required_cols = [
    "tsne_1",
    "tsne_2",
    label_col,
    patient_col
]

missing_cols = [
    col for col in required_cols
    if col not in df_merged.columns
]

if missing_cols:
    raise ValueError(f"Missing columns: {missing_cols}")

plot_df = df_merged.dropna(
    subset=required_cols
).copy()

plot_df[label_col] = (
    plot_df[label_col]
    .astype(str)
    .str.strip()
)

plot_df[patient_col] = (
    plot_df[patient_col]
    .astype(str)
    .str.strip()
)

# Standardize labels for reliable matching
plot_df["_label_clean"] = (
    plot_df[label_col]
    .str.lower()
    .str.replace("-", "", regex=False)
    .str.replace("_", "", regex=False)
    .str.replace(" ", "", regex=False)
)

aml_labels = {"aml"}
control_labels = {"control", "healthy"}
preleukemia_labels = {
    "mrd",
    "preleukemia",
    "preleukemic",
    "preleukaemia",
    "preleukaemic"
}

aml_df = plot_df[
    plot_df["_label_clean"].isin(aml_labels)
].copy()

control_df = plot_df[
    plot_df["_label_clean"].isin(control_labels)
].copy()

preleukemia_df = plot_df[
    plot_df["_label_clean"].isin(preleukemia_labels)
].copy()

if preleukemia_df.empty:
    available_labels = sorted(plot_df[label_col].unique())

    raise ValueError(
        "No preleukemic cells were found. "
        f"Available labels are: {available_labels}. "
        "Add the correct label to 'preleukemia_labels'."
    )

# Assign one color to each preleukemic patient
preleukemia_patients = sorted(
    preleukemia_df[patient_col].unique()
)

#n_colors = 16 
#patient_colors = plt.cm.plasma(np.linspace(0, 1, n_colors))

all_colors = plt.cm.tab20.colors
patient_colors = all_colors[:14] + all_colors[16:]

patient_color_map = {
    patient: patient_colors[i % len(patient_colors)]
    for i, patient in enumerate(preleukemia_patients)
}

fig, ax = plt.subplots(figsize=(4, 4))

# --------------------------------------------------
# Control cells: light grey
# --------------------------------------------------
ax.scatter(
    control_df["tsne_1"],
    control_df["tsne_2"],
    color="lightgrey",
    alpha=0.35,
    s=18,
    edgecolors="none",
    zorder=1
)

# --------------------------------------------------
# AML cells: dark grey
# --------------------------------------------------
ax.scatter(
    aml_df["tsne_1"],
    aml_df["tsne_2"],
    color="dimgray",
    alpha=0.35,
    s=18,
    edgecolors="none",
    zorder=2
)

# --------------------------------------------------
# Preleukemic cells: colored by patient
# --------------------------------------------------
for patient in preleukemia_patients:
    patient_df = preleukemia_df[
        preleukemia_df[patient_col] == patient
    ]

    patient_color = patient_color_map[patient]

    ax.scatter(
        patient_df["tsne_1"],
        patient_df["tsne_2"],
        color=patient_color,
        alpha=0.35,
        s=18,
        edgecolors="none",
        zorder=3
    )

# --------------------------------------------------
# Overall Control mean
# --------------------------------------------------
if not control_df.empty:
    control_mean_x = control_df["tsne_1"].mean()
    control_mean_y = control_df["tsne_2"].mean()

    ax.scatter(
        control_mean_x,
        control_mean_y,
        color="lightgrey",
        s=320,
        edgecolor="black",
        linewidth=1.8,
        marker="o",
        zorder=10,
        label="Control overall mean"
    )

# --------------------------------------------------
# Overall AML mean
# --------------------------------------------------
if not aml_df.empty:
    aml_mean_x = aml_df["tsne_1"].mean()
    aml_mean_y = aml_df["tsne_2"].mean()

    ax.scatter(
        aml_mean_x,
        aml_mean_y,
        color="dimgray",
        s=320,
        edgecolor="black",
        linewidth=1.8,
        marker="o",
        zorder=10,
        label="AML overall mean"
    )

# --------------------------------------------------
# Preleukemic patient means
# --------------------------------------------------
for patient in preleukemia_patients:
    patient_df = preleukemia_df[
        preleukemia_df[patient_col] == patient
    ]

    mean_x = patient_df["tsne_1"].mean()
    mean_y = patient_df["tsne_2"].mean()

    ax.scatter(
        mean_x,
        mean_y,
        color=patient_color_map[patient],
        s=260,
        edgecolor="black",
        linewidth=1.4,
        marker="s",
        zorder=11,
        label=str(patient)
    )

ax.set_xlabel("tSNE 1")
ax.set_ylabel("tSNE 2")
ax.set_title("tSNE colored by preleukemic patient")

#ax.set_xlim(-20, 20)
ax.set_ylim(*YLIM)

# Remove duplicate legend entries if necessary
handles, labels = ax.get_legend_handles_labels()
unique_legend = dict(zip(labels, handles))

ax.legend(
    unique_legend.values(),
    unique_legend.keys(),
    title="Mean positions",
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
    frameon=False,
    fontsize=8,
    title_fontsize=9,
    markerscale=0.55
)

plt.savefig(
    output_dir / "tsne_preleukemia_by_patient.pdf",
    #bbox_inches="tight",
    dpi=300
)

plt.show()